# Gradient Learnings Data Analytics Hackathon 2026
# Module 3 — Exploratory Data Analysis (EDA)
**Ecosystem:** Olist Brazilian E-Commerce Marketplace Diagnostic (2016–2018)  
**Analytical Grain Invariant:** 1 row = 1 `order_id` (99,441 rows strictly preserved)  
**Primary Dataset:** `data/processed/analytical_model.parquet` (80 feature columns)  

---

## 1. Executive Overview

This notebook executes a competition-grade, evidence-based Exploratory Data Analysis across approximately 100,000 commercial transactions from the Olist Brazilian marketplace.

### Analytical Tenets:
1. **Business-Question Driven:** Every chart and metric directly resolves an explicit operational or strategic inquiry.
2. **Strict Grain Preservation:** All platform-level calculations adhere to the validated 1-to-1 order grain.
3. **Non-Causal Evidence:** Observations, distributions, and correlations are documented rigorously without conflating association with causation.
4. **Actionable Insights:** Differentiates common baseline facts from high-impact, signature operational discoveries.


## 2. Problem Context & Environment Setup

Olist connects merchants across Brazil to major online sales channels. Leadership requires evidence on:
- How marketplace volume, revenue, and satisfaction have trended over time.
- How delivery SLA performance relates to review scores.
- How seller concentration and geographic distance interact with freight and fulfillment.
- What operational segments drive the platform's customer dissatisfaction.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display, Image, Markdown
except ImportError:
    def display(*args, **kwargs):
        for a in args:
            print(a)
    class Image:
        def __init__(self, filename=None):
            self.filename = filename

# Display configuration
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Verify processed analytical model
model_path = '../data/processed/analytical_model.parquet'
if not os.path.exists(model_path):
    # Fallback to local path if running from root
    model_path = 'data/processed/analytical_model.parquet'

df = pd.read_parquet(model_path)
print(f"Loaded Analytical Model: {df.shape[0]:,} rows x {df.shape[1]} columns")
assert df.shape[0] == 99441, "Order grain invariant violated!"


## 3. Executive KPI Baseline

Before diving into distributions, we establish and validate the baseline metrics across volume, finances, sentiment, fulfillment, and customer retention.


In [ ]:
# Load and display the validated KPI baseline table
kpi_path = '../outputs/tables/eda_kpi_baseline.csv'
if not os.path.exists(kpi_path):
    kpi_path = 'outputs/tables/eda_kpi_baseline.csv'

kpi_df = pd.read_csv(kpi_path)
display(kpi_df)


## 4. Marketplace Structure & Concentration

We examine the commercial composition of the marketplace: order status breakdown, seller concentration, and product category shares.

**Business Questions:**
- What proportion of transactions successfully complete delivery vs cancel or stall?
- How concentrated is marketplace revenue among sellers?
- Do a handful of product categories dominate total transaction volume?


In [ ]:
# Display Figure 2: Order Status Composition
fig2_path = '../outputs/figures/fig02_order_status_composition.png' if os.path.exists('../outputs/figures/fig02_order_status_composition.png') else 'outputs/figures/fig02_order_status_composition.png'
display(Image(filename=fig2_path))

# Display Figure 3: Seller Revenue Concentration (Pareto Curve)
fig3_path = '../outputs/figures/fig03_seller_concentration_pareto.png' if os.path.exists('../outputs/figures/fig03_seller_concentration_pareto.png') else 'outputs/figures/fig03_seller_concentration_pareto.png'
display(Image(filename=fig3_path))

# Display Figure 4: Category Volume vs Revenue Share
fig4_path = '../outputs/figures/fig04_category_volume_vs_revenue_share.png' if os.path.exists('../outputs/figures/fig04_category_volume_vs_revenue_share.png') else 'outputs/figures/fig04_category_volume_vs_revenue_share.png'
display(Image(filename=fig4_path))


**Key Observations:**
- **Completion Rate:** 97.02% of orders (96,478) are successfully delivered; cancellations represent just 0.63% (625 orders).
- **Seller Concentration:** Extreme Pareto concentration — the **top 10% of sellers generate 66.7% of GMV** and 62.2% of orders. The top 20% generate 84.1% of GMV.
- **Category Leadership:** The top 10 product categories account for 63.0% of all orders, led by Bed/Bath/Table, Health & Beauty, and Sports & Leisure.


## 5. Time & Seasonality Dynamics

We track monthly marketplace behavior across 25 operating months to test for growth-quality divergence.

**Business Question:**
Did rapid platform volume expansion degrade customer satisfaction or delivery SLAs over time?


In [ ]:
# Display Figure 1: Monthly Marketplace Growth & Quality Divergence
fig1_path = '../outputs/figures/fig01_monthly_marketplace_growth_divergence.png' if os.path.exists('../outputs/figures/fig01_monthly_marketplace_growth_divergence.png') else 'outputs/figures/fig01_monthly_marketplace_growth_divergence.png'
display(Image(filename=fig1_path))

# Display monthly time-series metrics
ts_path = '../outputs/tables/eda_time_series.csv' if os.path.exists('../outputs/tables/eda_time_series.csv') else 'outputs/tables/eda_time_series.csv'
ts_df = pd.read_csv(ts_path)
display(ts_df[ts_df['is_standard_window']][['purchase_year_month', 'total_orders', 'total_gmv', 'avg_review_score', 'low_review_rate_pct', 'late_rate_pct', 'avg_delivery_days']])


**Key Observations & Growth Divergence Test:**
- Monthly order volume grew over **8x** from 800 orders (Jan 2017) to 6,512 orders (Aug 2018).
- **No Chronic Secular Degradation:** Mean review score remained resilient around 4.10–4.20 stars across normal operating periods.
- **Acute Capacity Shock at Black Friday (Nov 2017):** Orders surged 63% month-over-month to 7,544. Carrier networks were overwhelmed, pushing the late delivery rate to 16.2% and temporarily dropping average review score to 3.82 stars. Within 90 days, fulfillment stabilized.


## 6. Customer Order Frequency & Repeat Behavior

We analyze customer repeat purchase behavior using the distinction between `customer_id` (transaction token) and `customer_unique_id` (individual person).

**Business Question:**
What does repeat purchase behavior look like, and do repeat buyers receive a better operational experience?


In [ ]:
# Display Figure 17: Repeat vs One-Time Customer Experience
fig17_path = '../outputs/figures/fig17_repeat_vs_onetime_customer_experience.png' if os.path.exists('../outputs/figures/fig17_repeat_vs_onetime_customer_experience.png') else 'outputs/figures/fig17_repeat_vs_onetime_customer_experience.png'
display(Image(filename=fig17_path))

# Repeat customer summary
cust_orders = df.groupby('customer_unique_id').agg(
    order_count=('order_id', 'count'),
    total_spend=('order_gmv', 'sum'),
    mean_review=('review_score', 'mean')
)
print(f"Total Unique Customers: {len(cust_orders):,}")
print(f"Repeat Buyers (>=2 orders): {(cust_orders['order_count'] > 1).sum():,} ({((cust_orders['order_count'] > 1).mean()*100):.2f}%)")
print(f"Orders from Repeat Buyers: {cust_orders[cust_orders['order_count'] > 1]['order_count'].sum():,} ({cust_orders[cust_orders['order_count'] > 1]['order_count'].sum()/len(df)*100:.2f}%)")
print(f"Max orders by a single customer: {cust_orders['order_count'].max()}")


**Key Observations:**
- **Low Repeat Rate:** Only **3.12% of unique customers (2,997)** placed repeat orders in the 2-year window, generating 6.38% of platform orders.
- **Experience Contrast:** Repeat buyers award slightly higher ratings (4.11 vs 4.08) and suffer marginally lower delay rates (7.4% vs 8.1%), but the difference is modest.
- **Business Implication:** Olist functions primarily as a customer acquisition engine rather than a high-retention direct destination.


## 7. Product Category Heterogeneity

We examine volume, pricing, freight, and satisfaction variations across product categories.

**Business Questions:**
- Are certain product categories notably weaker in review ratings?
- Do shipping delays inflict uniform or differentiated damage across categories?


In [ ]:
# Display Figure 15: Category Delay vs Review Sensitivity
fig15_path = '../outputs/figures/fig15_category_delay_vs_review_sensitivity.png' if os.path.exists('../outputs/figures/fig15_category_delay_vs_review_sensitivity.png') else 'outputs/figures/fig15_category_delay_vs_review_sensitivity.png'
display(Image(filename=fig15_path))

# Top 15 categories by order volume
cat_path = '../outputs/tables/eda_category_summary.csv' if os.path.exists('../outputs/tables/eda_category_summary.csv') else 'outputs/tables/eda_category_summary.csv'
cat_df = pd.read_csv(cat_path)
display(cat_df.head(15)[['dominant_category', 'order_count', 'gmv_share_pct', 'mean_item_price', 'freight_share_pct', 'mean_review_score', 'late_delivery_rate_pct']])


**Key Observations:**
- **Category Delay Vulnerability:** Furniture and bulky home items experience higher shipping delays (10–14% late), but customer review scores remain relatively resilient (3.95–4.05 stars).
- In contrast, gifting, apparel, and personal electronics suffer severe review penalties when shipments are delayed.


## 8. Seller Operational Performance

We benchmark seller operational excellence using statistical reliability thresholds ($N \ge 30$ orders).

**Business Question:**
Does operational performance vary substantially between individual sellers, and is high volume correlated with poor quality?


In [ ]:
seller_path = '../outputs/tables/eda_seller_summary.csv' if os.path.exists('../outputs/tables/eda_seller_summary.csv') else 'outputs/tables/eda_seller_summary.csv'
seller_df = pd.read_csv(seller_path)

rel_sellers = seller_df[seller_df['statistically_reliable']]
print(f"Total Sellers: {len(seller_df):,}")
print(f"Statistically Reliable Sellers (>=30 orders): {len(rel_sellers):,}")
print(f"Reliable Sellers - Mean Review Score: {rel_sellers['mean_review'].mean():.2f} (Min: {rel_sellers['mean_review'].min():.2f}, Max: {rel_sellers['mean_review'].max():.2f})")
print(f"Reliable Sellers - Late Delivery Rate: {rel_sellers['late_rate_pct'].mean():.2f}% (Min: {rel_sellers['late_rate_pct'].min():.2f}%, Max: {rel_sellers['late_rate_pct'].max():.2f}%)")

# Display top 10 highest-volume sellers
display(rel_sellers.sort_values(by='order_count', ascending=False).head(10)[['dominant_seller', 'order_count', 'total_gmv', 'seller_state', 'mean_review', 'late_rate_pct']])


## 9. Delivery Durations & Delay Severity (Core Pillar)

Delivery timing is the central axis of customer experience. We dissect the transit duration distribution, the promised estimate gap, and the severity of delivery delays.

**Business Questions:**
- What is the true distribution of transit duration from purchase to customer delivery?
- Are promised delivery dates systematically padded with a safety buffer?


In [ ]:
# Display Figure 5: Delivery Duration Distribution
fig5_path = '../outputs/figures/fig05_delivery_duration_distribution.png' if os.path.exists('../outputs/figures/fig05_delivery_duration_distribution.png') else 'outputs/figures/fig05_delivery_duration_distribution.png'
display(Image(filename=fig5_path))

# Display Figure 6: Delivery Delay Distribution & Conservative Buffer
fig6_path = '../outputs/figures/fig06_delivery_delay_distribution_and_buffer.png' if os.path.exists('../outputs/figures/fig06_delivery_delay_distribution_and_buffer.png') else 'outputs/figures/fig06_delivery_delay_distribution_and_buffer.png'
display(Image(filename=fig6_path))

# Display delivery distribution buckets table
deliv_path = '../outputs/tables/eda_delivery_distribution.csv' if os.path.exists('../outputs/tables/eda_delivery_distribution.csv') else 'outputs/tables/eda_delivery_distribution.csv'
deliv_df = pd.read_csv(deliv_path)
display(deliv_df)


**Key Observations:**
- **Median Transit:** The median order is delivered in **10.2 days**; 90% arrive within 23.2 days.
- **The Conservative Heuristic Buffer:** 92.9% of orders arrive before the estimated delivery date. The median package arrives **11.95 days ahead of schedule**.
- **The Psychology of Lateness:** Because promised estimates are already generous (~25-30 days), when an order breaches the promised date, the customer has waited over 3 weeks, making dissatisfaction intense.


## 10. Customer Review Sentiment & Polarization

We investigate the distribution of customer ratings and uncover the single most impactful CRM operational defect on the platform.

**Business Questions:**
- How does customer review sentiment degrade as delivery delays mount?
- What explains why over 8,000 reviews were created before the order was marked delivered?


In [ ]:
# Display Figure 7: Review Score Polarization
fig7_path = '../outputs/figures/fig07_review_score_distribution_polarization.png' if os.path.exists('../outputs/figures/fig07_review_score_distribution_polarization.png') else 'outputs/figures/fig07_review_score_distribution_polarization.png'
display(Image(filename=fig7_path))

# Display Figure 8: Review Score by Delay Severity Bucket
fig8_path = '../outputs/figures/fig08_review_score_by_delay_bucket.png' if os.path.exists('../outputs/figures/fig08_review_score_by_delay_bucket.png') else 'outputs/figures/fig08_review_score_by_delay_bucket.png'
display(Image(filename=fig8_path))

# Display Figure 9: The Survey Trigger Mechanism
fig9_path = '../outputs/figures/fig09_survey_trigger_pre_vs_post_delivery.png' if os.path.exists('../outputs/figures/fig09_survey_trigger_pre_vs_post_delivery.png') else 'outputs/figures/fig09_survey_trigger_pre_vs_post_delivery.png'
display(Image(filename=fig9_path))

# Review timing breakdown table
rev_path = '../outputs/tables/eda_review_distribution.csv' if os.path.exists('../outputs/tables/eda_review_distribution.csv') else 'outputs/tables/eda_review_distribution.csv'
rev_df = pd.read_csv(rev_path)
display(rev_df)


**Key Discovery — The Survey Trigger Defect (Signature Candidate):**
- **Non-Linear Dissatisfaction Inflection:** On-time orders average **4.29 stars**. Minor delays (1–3d) plummet to **2.87 stars**. Severe delays (>7d) drop to **1.62 stars** (82.7% negative reviews).
- **The CRM Mechanism:** Olist dispatches surveys when the estimated delivery date elapses. For orders delayed in transit (5,335 orders), customers are prompted to evaluate a package they have not yet received.
- **Result:** Average rating collapses to **1.98 stars** with **70.9% 1–2 star ratings**, directly causing **26.09% of all platform negative reviews**.


## 11. Payment Behavior & Financing Dynamics

We examine how Brazilian payment methods and installment options support order values.

**Business Question:**
How do payment instruments and installment depth relate to order basket size?


In [ ]:
# Display Figure 10: Dominant Payment Share & AOV
fig10_path = '../outputs/figures/fig10_payment_type_share_and_aov.png' if os.path.exists('../outputs/figures/fig10_payment_type_share_and_aov.png') else 'outputs/figures/fig10_payment_type_share_and_aov.png'
display(Image(filename=fig10_path))

# Display Figure 11: Financing Depth vs Ticket Value
fig11_path = '../outputs/figures/fig11_installments_vs_ticket_value.png' if os.path.exists('../outputs/figures/fig11_installments_vs_ticket_value.png') else 'outputs/figures/fig11_installments_vs_ticket_value.png'
display(Image(filename=fig11_path))

# Payment summary table
pay_path = '../outputs/tables/eda_payment_summary.csv' if os.path.exists('../outputs/tables/eda_payment_summary.csv') else 'outputs/tables/eda_payment_summary.csv'
pay_df = pd.read_csv(pay_path)
display(pay_df)


**Key Observations:**
- **Credit Card Dominance:** Credit cards fund **75.4% of orders** and **78.4% of platform value** (AOV R$ 165.7).
- **Boleto Bancário:** Serves as the primary cash alternative (19.4% of orders, AOV R$ 143.6).
- **Installment Scaling:** Ticket size scales directly with installments: single-payment credit card orders average R$ 98.7, scaling monotonically to R$ 421.3 for 10 installments.


## 12. Geographic Distribution & Corridor Friction

We model the spatial flow of goods across Brazil using cleaned geolocation centroids and great-circle Haversine distances.

**Business Questions:**
- Where are sellers located relative to customer demand across Brazil?
- How strongly does physical distance govern shipping transit duration?
- Which interstate logistical corridors suffer the highest delivery failure rates?


In [ ]:
# Display Figure 12: Regional Supply Dependency (São Paulo Dominance)
fig12_path = '../outputs/figures/fig12_geographic_flow_seller_to_customer_states.png' if os.path.exists('../outputs/figures/fig12_geographic_flow_seller_to_customer_states.png') else 'outputs/figures/fig12_geographic_flow_seller_to_customer_states.png'
display(Image(filename=fig12_path))

# Display Figure 13: Haversine Distance vs Delivery Duration
fig13_path = '../outputs/figures/fig13_haversine_distance_vs_delivery_duration.png' if os.path.exists('../outputs/figures/fig13_haversine_distance_vs_delivery_duration.png') else 'outputs/figures/fig13_haversine_distance_vs_delivery_duration.png'
display(Image(filename=fig13_path))

# Display Figure 14: Top High-Volume Interstate Corridors by Failure Rate
fig14_path = '../outputs/figures/fig14_regional_corridor_delay_heatmap.png' if os.path.exists('../outputs/figures/fig14_regional_corridor_delay_heatmap.png') else 'outputs/figures/fig14_regional_corridor_delay_heatmap.png'
display(Image(filename=fig14_path))

# State summary table
geo_path = '../outputs/tables/eda_geography_summary.csv' if os.path.exists('../outputs/tables/eda_geography_summary.csv') else 'outputs/tables/eda_geography_summary.csv'
geo_df = pd.read_csv(geo_path)
display(geo_df.head(10))


**Key Observations:**
- **São Paulo Logistical Hegemony:** SP sellers fulfill **70.3% of all platform transactions**, supplying 50% to 80% of orders in every single Brazilian state.
- **Distance Friction:** Haversine distance averages 595 km (median 433 km). Inter-regional shipments exceed 2,000 km, adding an average of 8 to 15 additional days of transit.
- **Chokepoint Corridors:** Interstate lines out of SP into Bahia (`SP -> BA`), Rio de Janeiro (`SP -> RJ`), and Ceará (`SP -> CE`) experience late delivery rates of 11.4% to 18.2%.


## 13. Multi-Variable Interactions & Relationships

We evaluate non-linear and bivariate interactions across logistics, economics, and sentiment.


In [ ]:
# Display Figure 16: Freight Share vs Review Satisfaction
fig16_path = '../outputs/figures/fig16_freight_share_vs_satisfaction.png' if os.path.exists('../outputs/figures/fig16_freight_share_vs_satisfaction.png') else 'outputs/figures/fig16_freight_share_vs_satisfaction.png'
display(Image(filename=fig16_path))

# Interaction candidate table
inter_path = '../outputs/tables/eda_interaction_candidates.csv' if os.path.exists('../outputs/tables/eda_interaction_candidates.csv') else 'outputs/tables/eda_interaction_candidates.csv'
inter_df = pd.read_csv(inter_path)
display(inter_df)


**Key Observations:**
- Freight cost share has only a **weak correlation (-0.065)** with review scores. Customers tolerate paying high freight when purchasing distant goods, but react severely to unexpected delays.
- Delay Days $	imes$ Survey Timing exhibits the highest interaction effect with customer dissatisfaction.


## 14. Anomaly & Outlier Analysis

We document and classify operational extremes and data logging quirks across the dataset.


In [ ]:
anom_path = '../outputs/tables/eda_anomaly_register.csv' if os.path.exists('../outputs/tables/eda_anomaly_register.csv') else 'outputs/tables/eda_anomaly_register.csv'
anom_df = pd.read_csv(anom_path)
display(anom_df)


## 15. Hypothesis Discovery Scan (H1 – H8)

We review the 8 initial research hypotheses against empirical EDA evidence:

| Hypothesis ID | Statement | EDA Evidence | Empirical Status |
| :---: | :--- | :--- | :---: |
| **H1** | Order growth degrades review scores over time. | Monthly review score oscillated stably (4.01-4.22); degradation was an acute capacity shock in Nov 2017. | **PARTIALLY SUPPORTED** |
| **H2** | Dissatisfaction drops non-linearly with delay severity. | On-time: 4.29; 1-3d late: 2.87; >7d late: 1.62. Severe non-linear cliff. | **SUPPORTED** |
| **H3** | Estimated delivery dates are systematically conservative. | 92.9% delivered early; median package arrives 11.95 days before estimate. | **SUPPORTED** |
| **H4** | Inter-regional orders suffer higher severe delay rates. | Cross-regional corridors (SP -> BA/AM) have 3x higher failure rates than SP -> SP. | **SUPPORTED** |
| **H5** | High freight does not guarantee superior speed or reviews. | Highest freight quintile takes 17.8d vs 10.4d for lowest quintile. | **SUPPORTED** |
| **H6** | Bulky categories have higher delay tolerance. | Holds for minor delays (1-3d), but collapses for severe delays (>7d). | **PARTIALLY SUPPORTED** |
| **H7** | Severe delay is primary driver of low reviews. | Delay severity Spearman rho = -0.334; price/installments rho < 0.05. | **SUPPORTED** |
| **H8** | Minority of segments accounts for disproportionate low reviews. | In-transit surveyed delayed orders (5.4% of orders) drive 26.1% of negative reviews. | **SUPPORTED** |


## 16. Finding Register & Differentiating Candidates

We compile the structured register of major discoveries classified by competitive advantage:


In [ ]:
find_path = '../outputs/tables/eda_finding_register.csv' if os.path.exists('../outputs/tables/eda_finding_register.csv') else 'outputs/tables/eda_finding_register.csv'
find_df = pd.read_csv(find_path)
display(find_df[['finding_id', 'classification', 'observation', 'business_relevance']])


## 17. Executive Summary & Recommended Next Modules

We conclude EDA with the multi-panel Executive Exploratory Dashboard, summarizing platform scale, customer sentiment, delivery SLAs, and regional supply dominance.


In [ ]:
# Display Figure 18: Executive Exploratory Dashboard
fig18_path = '../outputs/figures/fig18_executive_exploratory_dashboard.png' if os.path.exists('../outputs/figures/fig18_executive_exploratory_dashboard.png') else 'outputs/figures/fig18_executive_exploratory_dashboard.png'
display(Image(filename=fig18_path))


### Recommended Next Modules:
1. **Module 4 — Trusted KPI Layer:** Codify all validated KPIs into a reusable production library `kpis.py`.
2. **Module 5 — Feature Engineering:** Formalize non-linear delay severity bins, corridor route pairs, survey timing flags, and Haversine distance features.
3. **Module 6–15 — Deep Dive Analytical Modules:** Conduct formal statistical modeling on time-series decomposition, delivery SLAs, and geographic corridors.
4. **Module 17–18 — Root Cause Multivariate Modeling:** Run multivariate logistic regression with odds ratios and marginal effects to isolate the independent impact of delay severity vs survey trigger timing.

---
**Module 3 — Exploratory Data Analysis: COMPLETE**
